# Lane change and on-ramp merge with pdSTL

This notebook follows the progression of the [STLCG motion-planning example](https://github.com/Bkdogbey/stlcg/blob/master/examples/Motion%20planning.ipynb): visualize the environment, expose the dynamics, construct the temporal-logic task, initialize a control sequence, optimize it with gradients, and inspect the result. Here the state is a Gaussian belief, collision checks use relative ego–traffic beliefs, and execution uses receding-horizon planning.

We study both production scenarios:

- **Lane change:** move from the lower lane into the target lane while remaining on the road and avoiding traffic.
- **On-ramp merge:** complete the same target-lane dwell before the ego vehicle reaches the end of the taper.

The notebook calls the project implementation directly; it does not redefine the planner or pdSTL semantics.

## 1. Imports and experiment controls

Set `SELECTED` to inspect either scenario in the detailed single-window walkthrough. The final section executes both scenarios.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Open this notebook from inside the repository.')
sys.path.insert(0, str(ROOT / 'src'))

from models.rollouts import lane_rollout
from planning.environment import lane_local_window, lane_subformulas
from planning.runners import (
    _lane_initial_controls,
    _lane_initial_state,
    run_lane_change,
    setup_problem,
)
from utils import get_device, load_config
from visualization.animation import animate_mpc
from visualization.planning import (
    COLORS,
    _draw_ego_vehicle,
    _draw_environment,
    _ellipse,
    plot_lane_merge,
)

SCENARIOS = {
    'lane_change': ROOT / 'configs/scenarios/lane_change.yaml',
    'lane_merge': ROOT / 'configs/scenarios/lane_merge.yaml',
}
SELECTED = 'lane_change'
MAKE_GIFS = False  # Optional: GIF rendering is slower than planning.
OUTPUT_DIR = ROOT / 'outputs/experiments/lane_change_merge'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = get_device()
print(f'Device: {device}')

## 2. Load the editable scenario configurations

Geometry, traffic, uncertainty, task timing, limits, warm starts, and optimizer settings all come from YAML. Changing those files changes this notebook and the normal runners together.

In [ ]:
configs = {name: load_config(path) for name, path in SCENARIOS.items()}
problems = {
    name: setup_problem(cfg, device=device, with_environment=True)
    for name, cfg in configs.items()
}
rows = [
    '| scenario | horizon | dt | process std | task window | dwell | ramp |',
    '|---|---:|---:|---:|---|---:|---|',
]
for name, cfg in configs.items():
    rows.append(
        f"| {name.replace('_', ' ')} | {cfg['H']} | {cfg['dt']:.2f} s | "
        f"{cfg['q_std']:.3f} | {cfg['task']['start_window_seconds']} s | "
        f"{cfg['task']['dwell_seconds']:.1f} s | "
        f"{'yes' if cfg.get('ramp') else 'no'} |"
    )
display(Markdown('\n'.join(rows)))

## 3. Environments

The green band is the target lane. Red vehicles are uncertain traffic predictions, and the blue vehicle is the ego state. The merge additionally has a tapered non-drivable region.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), layout='constrained')
for axis, (name, problem) in zip(axes, problems.items()):
    moving = _draw_environment(axis, problem.env, lane=True)
    initial_state = _lane_initial_state(problem)
    _draw_ego_vehicle(axis, problem.env, initial_state[0][:2].cpu().numpy())
    cfg = configs[name]
    traffic_x = [vehicle['x0'] for vehicle in cfg['traffic']]
    right = max(traffic_x + [(cfg.get('ramp') or {}).get('end_x', 45.0)]) + 10
    axis.set_xlim(min(-15.0, min(traffic_x) - 8), right)
    axis.set_title(name.replace('_', ' ').title(), fontweight='bold')
display(fig)
plt.close(fig)

## 4. Double-integrator belief dynamics

The ego state and control are

$$x_t=[p_x,p_y,v_x,v_y]^\top, \qquad u_t=[a_x,a_y]^\top.$$

The Gaussian belief propagates through

$$\mu_{t+1}=A\mu_t+Bu_t, \qquad \Sigma_{t+1}=A\Sigma_tA^\top+Q.$$

Controls are generated from unconstrained optimizer variables with `tanh`, then the lane rollout applies the configured speed and longitudinal/lateral acceleration limits. Traffic uses the same constant-velocity transition with its own process covariance.

In [ ]:
problem = problems[SELECTED]
cfg = configs[SELECTED]
torch.manual_seed(cfg.get('seed', 0))
print('A =')
print(problem.dyn.A.cpu().numpy())
print('\nB =')
print(problem.dyn.B.cpu().numpy())
print('\nQ =')
print(problem.dyn.Q.cpu().numpy())
print(f"\nControl limit from dynamics: ±{problem.dyn.u_max:.2f} m/s²")
print(f"Speed limits: {cfg['speed_bounds']} m/s")
print(f"Acceleration limits: {cfg['accel_bounds']} m/s²")

## 5. Initial ego and traffic beliefs

The planner predicts distributions, not only mean trajectories. Relative collision uncertainty adds the independent ego and traffic covariances:

$$\mu^{rel}_{i,t}=\mu^{traffic}_{i,t}-\mu^{ego}_t, \qquad \Sigma^{rel}_{i,t}=\Sigma^{traffic}_{i,t}+\Sigma^{ego}_t.$$

In [ ]:
state = _lane_initial_state(problem)
ego_mean, ego_cov, streak, traffic_mean, traffic_cov, entry_step = state
print('Ego mean [px, py, vx, vy]:', ego_mean.cpu().numpy())
print('Ego covariance diagonal:', ego_cov.diag().cpu().numpy())
print('Current target-lane streak:', streak)
for vehicle, mean, covariance in zip(cfg['traffic'], traffic_mean, traffic_cov):
    print(
        f"{vehicle['name']:>12}: mean={mean.cpu().numpy()}, "
        f"cov_diag={covariance.diag().cpu().numpy()}"
    )

## 6. Warm-start controls and the initial belief rollout

As in the STLCG example, optimization starts from a structured guess. Here the guess accelerates laterally into the target lane and then reduces the lateral acceleration. It is only an initialization; the temporal formula determines the optimized behavior.

In [ ]:
warm_controls = _lane_initial_controls(problem)
traffic_q = cfg['traffic_q_std'] ** 2 * torch.eye(
    4, dtype=problem.dyn.B.dtype, device=problem.dyn.device
)
names = [vehicle['name'] for vehicle in cfg['traffic']]
accel_bounds = (
    cfg['accel_bounds']['longitudinal'],
    cfg['accel_bounds']['lateral'],
)
rollout = lane_rollout(
    problem.dyn,
    ego_mean,
    ego_cov,
    traffic_mean,
    traffic_cov,
    traffic_q,
    names,
    cfg['speed_bounds'],
    accel_bounds,
)
raw_controls = problem.planner._control_parameters(warm_controls)
initial_prediction = rollout(raw_controls)
applied = initial_prediction.aux['applied_controls'].detach().cpu().numpy()
time_u = cfg['dt'] * np.arange(len(applied))
fig, axis = plt.subplots(figsize=(8, 3.5), layout='constrained')
axis.step(time_u, applied[:, 0], where='post', color=COLORS['mean'], label='longitudinal')
axis.step(time_u, applied[:, 1], where='post', color=COLORS['planned'], label='lateral')
axis.set(xlabel='prediction time [s]', ylabel='acceleration [m/s²]', title='Warm-start controls')
axis.grid(color=COLORS['workspace'], alpha=0.16)
axis.legend()
display(fig)
plt.close(fig)

In [ ]:
mean = initial_prediction.aux['mean_trace'][0].detach().cpu().numpy()
covariance = initial_prediction.aux['cov_trace'][0].detach().cpu().numpy()
traffic = initial_prediction.aux['traffic_mean_trace'][0].detach().cpu().numpy()
fig, axis = plt.subplots(figsize=(12, 4.5), layout='constrained')
_draw_environment(axis, problem.env, lane=True)
axis.plot(mean[:, 0], mean[:, 1], color=COLORS['planned'], linewidth=2, label='warm-start ego mean')
for index in range(0, len(mean), 7):
    _ellipse(axis, mean[index], covariance[index])
for vehicle_index, vehicle in enumerate(cfg['traffic']):
    axis.plot(
        traffic[:, vehicle_index, 0],
        traffic[:, vehicle_index, 1],
        color=COLORS['traffic'],
        linestyle=':',
        label='traffic means' if vehicle_index == 0 else '_nolegend_',
    )
axis.set_xlim(mean[:, 0].min() - 5, max(mean[:, 0].max(), traffic[:, :, 0].max()) + 8)
axis.set_title(f"{SELECTED.replace('_', ' ').title()}: initial prediction", fontweight='bold')
axis.legend(loc='upper center', ncol=3)
display(fig)
plt.close(fig)

## 7. Construct the lane pdSTL task

For each planning window, the overall task is a disjunction over feasible dwell-start times. Each candidate requires safety until its completion time and continuous target-lane occupancy for the configured dwell duration:

$$\varphi = \bigvee_{s \in \mathcal S} \left[\mathbf G_{[0,s+d]}(\text{road} \land \text{collision-free}) \land \mathbf G_{[s,s+d]}(\text{target lane})\right].$$

For the merge, each candidate also requires the ego front to remain before the ramp end when the dwell completes. Collision-free means that the relative longitudinal and lateral safety envelopes do **not** overlap for every traffic vehicle.

In [ ]:
window = lane_local_window(problem.env, 0, ego_mean, cfg, streak=0)
formulae = lane_subformulas(window, cfg['H'])
for name, formula in formulae.items():
    interval = formula.probability_interval(initial_prediction.belief_trajectory)
    print(f"{name:>20}: [{interval[0].item():.4f}, {interval[1].item():.4f}]")
start_step, end_step = window.metadata['task']['start_end_steps']
dwell_steps = window.metadata['task']['dwell_steps']
latest_start = end_step - dwell_steps
print(
    f"\nOverall task combines {latest_start - start_step + 1} feasible dwell starts "
    f"from step {start_step} through {latest_start}."
)

## 8. Differentiable optimization of one planning window

The planner minimizes

$$J(v)=-w_\varphi\,\widetilde P_\downarrow(\varphi)+w_u\sum_t\lVert u_t\rVert^2+w_{\Delta u}\sum_t\lVert u_t-u_{t-1}\rVert^2,$$

where $u_t=u_{max}\tanh(v_t)$ and $\widetilde P_\downarrow$ is the smooth lower-bound surrogate. Gradients flow from the formula through atomic Gaussian probabilities, the belief rollout, and finally to every control variable. The hard interval is used for certification and stopping, not as the gradient objective.

In [ ]:
specification = formulae['overall']
gradient_probe = problem.planner._control_parameters(
    warm_controls
).detach().clone().requires_grad_(True)
probe_prediction = rollout(gradient_probe)
probe_score = specification.smooth_lower(
    probe_prediction.belief_trajectory, problem.planner._beta(0)
)
(-probe_score).backward()
print(f"Initial control-gradient norm: {gradient_probe.grad.norm().item():.6f}")
initial_plan = problem.planner.evaluate_controls(
    rollout, warm_controls, spec=specification
)
optimized_plan = problem.planner.optimize_window(
    rollout, spec=specification, init_guess=warm_controls
)
print(f"Initial hard interval:   {initial_plan.hard_interval}")
print(f"Optimized hard interval: {optimized_plan.hard_interval}")
print(f"Iterations: {len(optimized_plan.loss_history)}")
print(f"Planning time: {optimized_plan.planning_time:.3f} s")

In [ ]:
initial_mean = initial_plan.rollout.aux['mean_trace'][0].detach().cpu().numpy()
optimized_mean = optimized_plan.rollout.aux['mean_trace'][0].detach().cpu().numpy()
fig, (ax_map, ax_loss) = plt.subplots(1, 2, figsize=(13, 4.5), layout='constrained')
_draw_environment(ax_map, problem.env, lane=True)
ax_map.plot(initial_mean[:, 0], initial_mean[:, 1], color=COLORS['workspace'], linestyle='--', label='warm start')
ax_map.plot(optimized_mean[:, 0], optimized_mean[:, 1], color=COLORS['mean'], linewidth=2, label='optimized mean')
ax_map.set_xlim(optimized_mean[:, 0].min() - 5, optimized_mean[:, 0].max() + 12)
ax_map.set_title('One-window trajectory update')
ax_map.legend()
ax_loss.plot(
    np.arange(1, len(optimized_plan.loss_history) + 1),
    optimized_plan.loss_history,
    color=COLORS['score'],
)
ax_loss.set(xlabel='gradient iteration', ylabel='objective', title='Optimization history')
ax_loss.grid(color=COLORS['workspace'], alpha=0.16)
display(fig)
plt.close(fig)

## 9. From one window to receding-horizon execution

A single optimized prediction is not executed open-loop. At each simulated step the production runner:

1. builds the local task using the absolute time and any target-lane dwell credit;
2. predicts ego and traffic beliefs over the horizon;
3. optimizes the complete pdSTL task;
4. applies only the first bounded control;
5. samples process noise, updates traffic, and checks collision, road, deadline, or success;
6. shifts the remaining controls to warm-start the next window.

This is feedback through repeated belief updates and re-optimization. Set `live=True` in `run_lane_change` to watch candidate paths and certificates update while it runs.

In [ ]:
results = {}
for name, path in SCENARIOS.items():
    print(f"\nRunning {name.replace('_', ' ')}...")
    results[name] = run_lane_change(
        str(path), show=False, save=False, live=False
    )

In [ ]:
rows = [
    '| scenario | outcome | executed steps | final certified interval |',
    '|---|---|---:|---|',
]
for name, result in results.items():
    interval = result.window_plans[-1].hard_interval
    rows.append(
        f"| {name.replace('_', ' ')} | {result.stopped_reason} | "
        f"{len(result.window_plans)} | [{interval[0]:.4f}, {interval[1]:.4f}] |"
    )
display(Markdown('\n'.join(rows)))

## 10. Inspect both executed plans and their certificates

The combined production figure shows the road execution, representative prediction windows, the per-window hard pdSTL interval, the smooth optimization score, and applied controls. The trajectory figure provides a cleaner road-scale view.

In [ ]:
for name, result in results.items():
    views = plot_lane_merge(
        result,
        problems[name].env,
        dt=configs[name]['dt'],
        save_path=OUTPUT_DIR / f'{name}.png',
        show=False,
    )
    display(Markdown(f"### {name.replace('_', ' ').title()}"))
    display(views['combined'][0])
    display(views['trajectory'][0])

## 11. Optional execution animations

Set `MAKE_GIFS = True` at the top to render both executions. Each frame shows the executed state, current prediction, moving traffic, applied control, and current certified interval.

In [ ]:
if MAKE_GIFS:
    for name, result in results.items():
        gif_path = OUTPUT_DIR / f'{name}.gif'
        animate_mpc(
            result,
            problems[name].env,
            dt=configs[name]['dt'],
            filename=gif_path,
            lane=True,
            show=False,
        )
        display(Image(filename=str(gif_path)))
else:
    print('Set MAKE_GIFS = True and rerun this cell to render animations.')

## What to change next

Use the YAML files rather than editing notebook logic:

- move traffic with `x0`, `y`, and `speed`;
- change uncertainty with `q_std`, `traffic_q_std`, and covariance scales;
- change the allowed completion window and dwell duration under `task`;
- reshape the on-ramp with `ramp.start_x` and `ramp.end_x`;
- change physical limits under `speed_bounds` and `accel_bounds`;
- change optimization behavior under `planner`.

After any change, rerun from the configuration cells so the environment, formula timing, rollout, optimizer, and execution remain consistent.